In [1]:
import pandas as pd

In [2]:
# load tf list
fpath = "allTFs_hg38.txt"
tf_list = [x.strip() for x in open(fpath)]
print(f"{len(tf_list)=}")
tf_list[:10]

len(tf_list)=1892


['ZNF354C',
 'KLF12',
 'ZNF143',
 'ZIC2',
 'ZNF274',
 'SP2',
 'ZBTB7A',
 'BCL6B',
 'ZBTB49',
 'ZIC1']

In [3]:
def load_pathway(fpath):
    """
    Loads an Enrichr-like database file into a boolean DataFrame.

    Args:
        fpath (str): Path to the Enrichr-like database file.

    Returns:
        pandas.DataFrame: A boolean DataFrame where:
            - Index: Genes
            - Columns: Pathways
            - Values: True if the gene is in the pathway, False otherwise.
    """

    result = []
    with open(fpath,  encoding='utf-8') as f:
        for line in f:
            split_line = [x for x in line.strip().split('\t') if x]  # Remove empty strings directly

            row = {'label': split_line[0]}
            for gene in split_line[1:]:
                row[gene] = 1

            result.append(row)

    df = pd.DataFrame(result)
    df = df.fillna(0.0).set_index('label').astype(bool).T  # Chained operations for clarity
    return df

In [4]:
fpaths = {
    'panglao' : "PanglaoDB_Augmented_2021.txt",
    'tabula' : "Tabula_Sapiens.txt",
}

df = []

for key, fpath in fpaths.items():
    tab = load_pathway(fpath)
    tab = tab.astype(int)
    tab = tab.reset_index(names=['gene_name'])
    tab = pd.melt(tab, id_vars='gene_name')

    # drop non-markers
    tab = tab[tab['value'] > 0]
    tab['is_tf'] = tab['gene_name'].isin(tf_list)
    tab = tab.drop(columns='value')
    tab = tab.rename(columns={'label' : 'cell_type'})
    tab['cell_type'] = tab['cell_type'].str.lower().str.replace(" ", "_")
    tab['source'] = key
    print(f"{tab.shape=}")
    df.append(tab)

df = pd.concat(df)
print(f"{df.shape=}")
df.head()

tab.shape=(24223, 4)
tab.shape=(46866, 4)
df.shape=(71089, 4)


,gene_name,cell_type,is_tf,source
0,GDF15,acinar_cells,False,panglao
1,RARRES2,acinar_cells,False,panglao
2,TM4SF4,acinar_cells,False,panglao
3,CELA1,acinar_cells,False,panglao
4,GCG,acinar_cells,False,panglao


In [5]:
cell_type_categories = {
    "fibroblast": [
        "fibroblasts",
        "bladder-fibroblast",
        "large_intestine-fibroblast",
        "lung-fibroblast",
        "pancreas-fibroblast",
        "prostate-fibroblast",
        "salivary_gland-fibroblast",
        "tongue-fibroblast",
        "uterus-fibroblast",
        "vasculature-fibroblast",
        "fat-fibroblast",
        "eye-fibroblast",
        "heart-fibroblast_of_cardiac_tissue",
        "mammary-fibroblast_of_breast"
    ],
    "endothelial": [
        "endothelial_cells",
        "endothelial_cells_(aorta)",
        "endothelial_cells_(blood_brain_barrier)",
        "bladder-capillary_endothelial_cell",
        "bladder-endothelial_cell_of_lymphatic_vessel",
        "bladder-vein_endothelial_cell",
        "lung-capillary_endothelial_cell",
        "lung-endothelial_cell_of_artery",
        "lung-endothelial_cell_of_lymphatic_vessel",
        "lung-lung_microvascular_endothelial_cell",
        "lung-vein_endothelial_cell",
        "muscle-capillary_endothelial_cell",
        "muscle-endothelial_cell_of_artery",
        "muscle-endothelial_cell_of_lymphatic_vessel",
        "muscle-endothelial_cell_of_vascular_tree",
        "pancreas-endothelial_cell",
        "kidney-endothelial_cell",
        "large_intestine-gut_endothelial_cell",
        "small_intestine-gut_endothelial_cell",
        "spleen-endothelial_cell",
        "thymus-capillary_endothelial_cell",
        "thymus-endothelial_cell_of_artery",
        "thymus-endothelial_cell_of_lymphatic_vessel",
        "thymus-vein_endothelial_cell",
        "trachea-endothelial_cell",
        "vasculature-endothelial_cell",
        "vasculature-artery_endothelial_cell",
        "vasculature-lymphatic_endothelial_cell",
        "liver-endothelial_cell",
        "liver-endothelial_cell_of_hepatic_sinusoid",
        "eye-retinal_blood_vessel_endothelial_cell",
        "mammary-endothelial_cell_of_artery",
        "mammary-endothelial_cell_of_lymphatic_vessel",
        "mammary-vein_endothelial_cell",
        "salivary_gland-endothelial_cell",
        "salivary_gland-endothelial_cell_of_lymphatic_vessel",
        "uterus-endothelial_cell",
        "uterus-endothelial_cell_of_lymphatic_vessel",
        "tongue-capillary_endothelial_cell",
        "tongue-endothelial_cell_of_artery",
        "tongue-endothelial_cell_of_lymphatic_vessel",
        "tongue-vein_endothelial_cell",
        "prostate-endothelial_cell",
        "skin-endothelial_cell",
        "fat-endothelial_cell",
        "eye-endothelial_cell"
    ],
    "hematopoietic": [
        "hematopoietic_stem_cells",
        "blood-hematopoietic_stem_cell",
        "bone_marrow-hematopoietic_stem_cell",
        "lymph_node-hematopoietic_stem_cell",
        "spleen-hematopoietic_stem_cell",
        "erythroid-like_and_erythroid_precursor_cells",
        "blood-myeloid_progenitor",
        "bone_marrow-myeloid_progenitor",
    ]
}

# Reverse mapping: each cell type → its category
cell_type_to_category = {
    cell_type: category
    for category, cell_types in cell_type_categories.items()
    for cell_type in cell_types
}

df['category'] = df['cell_type'].map(cell_type_to_category)
df = df[df['category'].notna()]

print(df['category'].value_counts().to_string())

category
endothelial      4841
fibroblast       1532
hematopoietic     931


In [8]:
outpath = "clean_marker_genes.csv"
df = df.reset_index(drop=True)
df.to_csv(outpath, index=False)
df.head()

,gene_name,cell_type,is_tf,source,category
0,TSPAN8,endothelial_cells,False,panglao,endothelial
1,KLK1,endothelial_cells,False,panglao,endothelial
2,RNASE1,endothelial_cells,False,panglao,endothelial
3,LUM,endothelial_cells,False,panglao,endothelial
4,LOX,endothelial_cells,False,panglao,endothelial
